<div style="direction: rtl !important; text-align: right !important; font-family: inherit;">

<h2 style="direction: rtl !important; text-align: right !important; font-family: inherit; font-size: 1.5em; font-weight: bold; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 5px;">בדיקת היתכנות והמחשת מגבלות ה OCR </h2>

<p style="direction: rtl !important; text-align: right !important; font-family: inherit; line-height: 1.6;">
כשלב ראשוני ומקדים, יצרנו מספר קבצים מסוג PDF ו-Word המכילים דרכונים תקינים לצד מקרי קצה שונים ומצבים לא אופטימליים.
</p>

<p style="direction: rtl !important; text-align: right !important; font-family: inherit; line-height: 1.6;">
<b>המטרה:</b> להריץ על קבצים אלו מנוע OCR סטנדרטי ולהמחיש באופן מעשי היכן הוא נכשל (למשל בתנאי תאורה בעייתים, זוויות או רעש ברקע). שלב זה נועד להמחשת הבעיה ודחיקת הגבולות של הכלים הקיימים.
</p>

</div>

In [1]:
import cv2
import random
import numpy as np
import os
import io
from docx import Document
from docx.shared import Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH

# Source files
source_image_1 = 'data/passports_images/passport1.jpg'
source_image_2 = 'data/passports_images/passport2.jpg'
user_extreme_source = 'data/passports_images/passport_on_table.png'

# Pool of words to generate realistic random text
TECH_WORDS = [
    "data", "model", "validation", "document", "scan", "system", 
    "process", "image", "resolution", "pixel", "accuracy", "report", 
    "network", "layer", "feature", "output", "matrix", "analysis"
]

def generate_random_paragraph():
    """Generates a single paragraph consisting of several random sentences"""
    sentence_count = random.randint(3, 6)
    sentences = []
    for _ in range(sentence_count):
        word_count = random.randint(7, 12)
        words = [random.choice(TECH_WORDS) for _ in range(word_count)]
        sentences.append(" ".join(words).capitalize() + ".")
    return " ".join(sentences)

def apply_isolated_distortion(base_image_path, mode):
    if not os.path.exists(base_image_path):
        raise FileNotFoundError(f"CRITICAL ERROR: Could not find '{base_image_path}'.")
        
    img = cv2.imread(base_image_path)
    if img is None:
        raise ValueError(f"CRITICAL ERROR: OpenCV could not read '{base_image_path}'.")
        
    # Define angles
    angles = {
        'cropped_only': 0, 'blur': -15, 'noise': 20,
        'photocopy': -25, 'pixelated': 15, 'grayscale': 35
    }
    angle = angles.get(mode, 0)
    
    # Apply rotation only if needed
    if angle != 0:
        h_img, w_img = img.shape[:2]
        M = cv2.getRotationMatrix2D((w_img // 2, h_img // 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w_img, h_img), borderValue=(245, 245, 245))

    # Apply specific effect
    if mode == 'blur':
        img = cv2.GaussianBlur(img, (27, 27), 0)
    elif mode == 'noise':
        noise = np.zeros(img.shape, np.int16)
        cv2.randn(noise, 0, 130)
        img = cv2.add(img, noise, dtype=cv2.CV_8UC3) 
    elif mode == 'photocopy':
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        img = cv2.convertScaleAbs(img, alpha=2.5, beta=-50)
    elif mode == 'pixelated':
        h_c, w_c = img.shape[:2]
        img = cv2.resize(img, (max(1, w_c // 5), max(1, h_c // 5)), interpolation=cv2.INTER_LINEAR)
        img = cv2.resize(img, (w_c, h_c), interpolation=cv2.INTER_NEAREST)
    elif mode == 'grayscale':
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Encode to memory
    is_success, buffer = cv2.imencode(".jpg", img)
    if not is_success:
        raise ValueError("CRITICAL ERROR: Failed to encode image to memory.")
    
    return io.BytesIO(buffer)

def apply_normal_upside_down(base_image_path):
    if not os.path.exists(base_image_path):
        raise FileNotFoundError(f"CRITICAL ERROR: Could not find '{base_image_path}'.")
    
    img = cv2.imread(base_image_path)
    if img is None:
        raise ValueError(f"CRITICAL ERROR: OpenCV could not read '{base_image_path}'.")
    
    # Random 180-degree flip (no cropping)
    if random.choice([True, False]):
        img = cv2.rotate(img, cv2.ROTATE_180)
    
    # Encode to memory
    is_success, buffer = cv2.imencode(".jpg", img)
    if not is_success:
        raise ValueError("CRITICAL ERROR: Failed to encode image to memory.")
    return io.BytesIO(buffer)

def apply_maksim_layout(doc, image_stream, effect_name=None):
    # Add heading
    doc.add_heading(f'Application Form - {random.randint(1000, 9999)}', level=1)
    
    if effect_name:
        doc.add_paragraph(f"Effect: {effect_name}")
        
    # Generate a large number of paragraphs to create multiple pages
    total_paragraphs = random.randint(15, 30)
    
    # Randomly determine the exact position where the image will appear
    image_insert_position = random.randint(2, total_paragraphs - 2)

    for i in range(total_paragraphs):
        # Add a paragraph of random text
        doc.add_paragraph(generate_random_paragraph())
        
        # If the current index matches the random position, insert the image
        if i == image_insert_position:
            doc.add_paragraph("--- DOCUMENT SCAN BEGINS ---")
            
            if isinstance(image_stream, io.BytesIO):
                image_stream.seek(0)
            
            img_paragraph = doc.add_paragraph()
            run = img_paragraph.add_run()
            random_width = random.uniform(8.0, 15.0) 
            run.add_picture(image_stream, width=Cm(random_width))
            
            # Random alignment for the image
            alignments = [WD_ALIGN_PARAGRAPH.LEFT, WD_ALIGN_PARAGRAPH.CENTER, WD_ALIGN_PARAGRAPH.RIGHT]
            img_paragraph.alignment = random.choice(alignments)
            
            doc.add_paragraph("--- DOCUMENT SCAN ENDS ---")

def create_doc_with_image(doc_name, image_stream, effect_name):
    doc = Document()
    apply_maksim_layout(doc, image_stream, effect_name)
    doc.save(doc_name)

def create_normal_doc(doc_name):
    doc = Document()
    chosen_img_path = random.choice([source_image_1, source_image_2])
    stream = apply_normal_upside_down(chosen_img_path)
    apply_maksim_layout(doc, stream)
    doc.save(doc_name)

# Generate normal documents (1-10)
for i in range(1, 11):
    create_normal_doc(f'Word_Normal_Doc_{i}.docx')

# Generate edge case documents (11-16)
modes = ['cropped_only', 'blur', 'noise', 'photocopy', 'pixelated', 'grayscale']
for i, mode in enumerate(modes, start=11):
    memory_image = apply_isolated_distortion(user_extreme_source, mode)
    doc_name = f'Word_Edge_Case_{i}.docx'
    create_doc_with_image(doc_name, memory_image, mode)
    
print("Process completed successfully! Word documents were created.")

Process completed successfully! Word documents were created.


<div style="direction: rtl !important; text-align: right !important; font-family: inherit;">

<h2 style="direction: rtl !important; text-align: right !important; font-family: inherit; font-size: 1.5em; font-weight: bold; color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 5px;">יצירת מאגר מסמכים סינתטי </h2>

<p style="direction: rtl !important; text-align: right !important; font-family: inherit; line-height: 1.6;">
בהמשך למוטיבציה שהוצגה בפתח הפרויקט  עולה הצורך ליצור ולדמות מסמכים מורכבים ולא אופטימליים,  בחלק זה אנו מיישמים סקריפט אוטומטי ליצירת מאגר מסמכי word, מאגר זה ישמש כסביבת בדיקה מבוקרת להערכת הביצועים של מנוע ה OCR הסטנדרטי.
</p>

<h3 style="direction: rtl !important; text-align: right !important; font-family: inherit; font-size: 1.2em; font-weight: bold; color: #2c3e50; margin-top: 15px;">מבנה הסקריפט והכלים המרכזיים</h3>
<ul style="direction: rtl !important; text-align: right !important; font-family: inherit; line-height: 1.6; margin-right: 20px;">
    <li><b>הדמיית מסמך ארגוני:</b> יצירת פסקאות טקסט אקראיות כדי למקם את תמונת הדרכון בתוך מסמך רב-עמודי אמיתי, ולא כתמונה בודדת.</li>
    <li><b>הזרקת רעשים (OpenCV & NumPy):</b> שימוש ב-<code>cv2</code> להחלת מניפולציות חזותיות על הדרכון (טשטוש, רעש, סיבובים והדמיית צילום מסמך) כדי לאתגר את מנוע הזיהוי.</li>
    <li><b>בנייה אוטומטית של קבצי Word (python-docx):</b> אריזת התמונות המעובדות בתוך קבצי <code>.docx</code>, תוך קביעה אקראית של מיקום התמונה, גודלה והיישור שלה בעמוד.</li>
</ul>

<h3 style="direction: rtl !important; text-align: right !important; font-family: inherit; font-size: 1.2em; font-weight: bold; color: #2c3e50; margin-top: 15px;">המטרה המתודולוגית</h3>
<p style="direction: rtl !important; text-align: right !important; font-family: inherit; line-height: 1.6;">
  שלב זה מהווה קו בסיס (Baseline), הרצת כלי ה-OCR על מאגר זה תמחיש אמפירית היכן הם נכשלים ב"תנאי אמת", ובכך תספק את ההצדקה לניתוח הכישלונות ולמעבר לפתרון מתקדם יותר.
</p>

</div>